# 结构化掩码可视化：Det / COCO（Mask R-CNN R50-FPN + EDEN）
**回答的问题：各训练阶段的 ckpt 当前会怎么蒙。**

口径与其他三臂一致：G = |∂L_task/∂x₀|（task loss = RPN + RoI 损失求和，
复刻 MaskRCNNWithDiffAE.loss 的任务侧，不含 DiffAE 分支）-> 划块均值 ->
softmax 概率 -> Gumbel-top-k 现采 -> 蒙后输入 x̃₀。CRN：跨 ckpt 同图同噪声。

mmdet val_dataloader 默认 batch_size=1 且各图 padding 尺寸不一——取数跨
batch 累积、逐图反传（batch_grad=False）。
需在 mmdetection 环境下运行（det_probe 负责注册 MaskRCNNWithDiffAE）。

In [ ]:
import os, sys
import torch

# ---- 先在 notebook 所在目录（visual/）下把路径算成绝对路径 ----
VISUAL = os.path.abspath('.')
MMDET  = os.path.abspath('../tasks/Detection/mmdetection')

os.chdir(MMDET)
for p in [MMDET, os.path.join(MMDET, 'eden'), VISUAL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)

import importlib
import probe_core as pc
import det_probe as dp
importlib.reload(pc)
importlib.reload(dp)
assert getattr(pc, 'MASK_VIZ_VERSION', 0) >= 1, \
    'probe_core 是旧版（无掩码可视化）——请确认 visual/probe_core.py 已更新并重启内核'

In [ ]:
# ---- 配置：改成你的 run 目录 ----
# CFG 用 run 目录里 train 时 dump 的完整 config（口径与训练一致）。
import glob, re
RUN_DIR = 'work_dirs/eden/maskrcnn_diffae_tweight_mask'
CFG = sorted(glob.glob(os.path.join(RUN_DIR, '*.py')))[0]
CKPTS = sorted(glob.glob(f'{RUN_DIR}/epoch_*.pth') + glob.glob(f'{RUN_DIR}/iter_*.pth'),
               key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
OUT = 'work_dirs/offline_probe/mask_viz'
NUM_IMAGES = 8
DISPLAY_IMAGES = 4
MASK_RATIO = None     # None = 模型配置值
GUIDED = True         # True = G 引导选址；False = 预热期均匀随机
SEED = 0
os.makedirs(OUT, exist_ok=True)
CKPTS

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
model, cfg = dp.build_det_model(CFG)   # 先建一次只为取数
batch = dp.build_det_val_batch(model, cfg, NUM_IMAGES)
del model; torch.cuda.empty_cache()

In [ ]:
results, labels = [], []
for ckpt in CKPTS:
    model, _ = dp.build_det_model(CFG, ckpt)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    # det 逐图反传（各图 padding 尺寸不一）：task loss = RPN + RoI 损失求和
    grad_fn = dp.make_det_grad_fn(model)
    res = pc.measure_mask_viz(model, batch, grad_fn, batch_grad=False,
                              mask_ratio=MASK_RATIO, guided=GUIDED, seed=SEED)
    pc.save_mask_viz_cache(os.path.join(OUT, f'{tag}_cache.npz'), res)
    fig = pc.plot_mask_viz(res, title=tag, max_images=DISPLAY_IMAGES,
                           save_path=os.path.join(OUT, f'{tag}_mask_viz.png'))
    results.append(res); labels.append(tag)
    print(f'{tag}: masked={res["masked_frac"].mean():.2f} '
          f'(k={res["k"]}/{res["g"]**2}, alpha={res["alpha"]:g})')
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 演化拼图：行 = 训练阶段，列 = 同一批图
for mode in ['masked', 'prob', 'g']:
    pc.plot_mask_viz_evolution(results, labels, mode=mode,
                               max_images=DISPLAY_IMAGES,
                               save_path=os.path.join(OUT, f'evolution_{mode}.png'))
print('saved evolution_{masked,prob,g}.png')

In [ ]:
# 免重测复画：从 npz 缓存重建 res（换 DISPLAY_IMAGES / 标题都行；掩码已在缓存里）
REPLAY_TAG = os.path.splitext(os.path.basename(CKPTS[-1]))[0]  # 改成想复画的 ckpt tag
res = pc.load_mask_viz_cache(os.path.join(OUT, f'{REPLAY_TAG}_cache.npz'))
fig = pc.plot_mask_viz(res, title=f'{REPLAY_TAG} (replay)',
                       max_images=DISPLAY_IMAGES,
                       save_path=os.path.join(OUT, f'{REPLAY_TAG}_replay.png'))
fig